In [1]:
import pandas as pd
import numpy as np
from sklearn.multioutput import ClassifierChain, MultiOutputClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
import xgboost as xgb
from sklearn.metrics import f1_score, hamming_loss, accuracy_score
from sklearn.preprocessing import StandardScaler
import time
from pathlib import Path

# ==========================================
# 1. SETUP & LOAD DATA
# ==========================================
root_path = Path.cwd().parent
print("🔍 1. Memuat Data untuk Benchmarking...")

train_df = pd.read_csv(root_path / "Data/processed/train_balanced_multilabel.csv")
test_df = pd.read_csv(root_path / "Data/split/test_data.csv")

# Menggunakan 35 fitur hasil seleksi B-MFO
features = ['TIPI1', 'TIPI2', 'TIPI3', 'TIPI4', 'TIPI5', 'TIPI6', 'TIPI7', 'TIPI8', 'TIPI9', 'TIPI10', 
            'VCL1', 'VCL2', 'VCL3', 'VCL4', 'VCL5', 'VCL7', 'VCL8', 'VCL9', 'VCL10', 'VCL11', 'VCL12', 
            'VCL13', 'VCL14', 'VCL15', 'VCL16', 'education', 'urban', 'gender', 'engnat', 'age', 
            'religion', 'orientation', 'race', 'voted', 'married']
targets = ['risk_depression', 'risk_anxiety', 'risk_stress']

X_train, Y_train = train_df[features], train_df[targets]
X_test, Y_test = test_df[features], test_df[targets]

# SVM sangat sensitif terhadap skala data, jadi kita lakukan Scaling khusus untuk SVM
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# ==========================================
# 2. DEFINISI MODEL BENCHMARK
# ==========================================
print("🥊 2. Mempersiapkan Tiga Arsitektur Berbeda...")

models = {
    # Main Model: B-MFO + XGBoost + Classifier Chains
    "B-MFO + XGBoost (Chains)": ClassifierChain(
        xgb.XGBClassifier(n_estimators=200, learning_rate=0.05, max_depth=5, random_state=42),
        order='random', random_state=42
    ),
    
    # Baseline 1: Random Forest + Binary Relevance (OvR)
    "RF (Binary Relevance)": MultiOutputClassifier(
        RandomForestClassifier(n_estimators=100, random_state=42)
    ),
    
    # Baseline 2: SVM + Binary Relevance (OvR)
    "SVM (Binary Relevance)": MultiOutputClassifier(
        SVC(probability=True, random_state=42)
    )
}

# ==========================================
# 3. EKSEKUSI & EVALUASI
# ==========================================
benchmarking_results = []

for name, model in models.items():
    print(f"⏳ Mengevaluasi {name}...")
    start_time = time.time()
    
    # Gunakan data scaled khusus untuk SVM
    X_tr = X_train_scaled if "SVM" in name else X_train
    X_te = X_test_scaled if "SVM" in name else X_test
    
    model.fit(X_tr, Y_train)
    predictions = model.predict(X_te)
    
    exec_time = time.time() - start_time
    
    # Hitung Metrik
    results = {
        "Model": name,
        "Macro F1": f1_score(Y_test, predictions, average='macro'),
        "Hamming Loss": hamming_loss(Y_test, predictions),
        "Exact Match": accuracy_score(Y_test, predictions),
        "Time (s)": exec_time
    }
    benchmarking_results.append(results)

# ==========================================
# 4. TABEL PERBANDINGAN FINAL
# ==========================================
df_results = pd.DataFrame(benchmarking_results).sort_values(by="Macro F1", ascending=False)
print("\n" + "="*70)
print("📊 HASIL BENCHMARKING MODEL AKHIR")
print("="*70)
print(df_results.to_string(index=False))
print("="*70)

🔍 1. Memuat Data untuk Benchmarking...
🥊 2. Mempersiapkan Tiga Arsitektur Berbeda...
⏳ Mengevaluasi B-MFO + XGBoost (Chains)...
⏳ Mengevaluasi RF (Binary Relevance)...
⏳ Mengevaluasi SVM (Binary Relevance)...

📊 HASIL BENCHMARKING MODEL AKHIR
                   Model  Macro F1  Hamming Loss  Exact Match  Time (s)
B-MFO + XGBoost (Chains)  0.783471      0.273301     0.560706  1.871568
   RF (Binary Relevance)  0.781029      0.272443     0.529433  3.692039
  SVM (Binary Relevance)  0.773261      0.279004     0.525938 68.708433
